# Tool Calling (Phase 7 — Week 7)

#### Where we are

In Phase 6 you saw **function calling**: the model can return a structured request like

```
get_weather(city="Chennai")
```

But it stopped there — the model only *asked*. Nothing actually ran.

**Tool calling** completes the circle:

```
User
 ↓
LLM            "I need the weather. Call get_weather('Chennai')."
 ↓
Tool Request
 ↓
Weather API    your backend runs the real function
 ↓
LLM            "It's 31°C and sunny."  (model reads the result)
 ↓
Answer         "It's currently 31°C and sunny in Chennai."
```

The key new idea:

```
The LLM doesn't run tools.
YOUR code runs tools and feeds the result back to the LLM.
```

By the end, you'll understand the sentence that unlocks Phase 9:

***An agent is basically an LLM that can use tools.***

### Why does an LLM need tools?

An LLM, on its own, has hard limits:

```
✗ No real-time data     →  "What's the weather right now?"
✗ Unreliable math       →  big multiplications, precise calculations
✗ No access to YOUR data→  "How many orders did customer 42 place?"
✗ Can't take actions    →  send an email, create a ticket
```

It only predicts text. Tools fix this by letting it **reach the outside world**:

```
LLM  →  decides WHAT to do
Tool →  actually DOES it (API, DB, calculator, code)
```

#### The mental shift

```
Without tools:  the model GUESSES from training data (risk: hallucination)

With tools:     the model FETCHES real facts and acts on them
```

This is also a major hallucination defense (Phase 1): instead of inventing an answer, the model calls a tool and uses the real result.

### Anatomy of a Tool

A tool is just a normal function plus a **description the model can read**.

```
name        →  get_weather
description →  "Get the current weather for a city"
parameters  →  { city: string, unit: "celsius" | "fahrenheit" }
the code    →  the actual Python function you wrote
```

```python
def get_weather(city: str, unit: str = "celsius") -> dict:
    # calls a real weather API
    return {"temp": 31, "condition": "sunny", "unit": unit}
```

#### The description is not decoration — it's the interface

The model decides **whether** and **how** to call a tool based on its `name`, `description`, and parameter names.

```
Vague:  "weather"                 → model may misuse it
Clear:  "Get current weather for  → model calls it correctly
         a given city name"
```

```
Tip: write tool descriptions like API docs for a junior dev.
     Clear name, clear purpose, clear parameters.
```

### The Tool-Calling Loop (step by step)

```
1. You send: the user question + the list of available tools.

2. The model replies with EITHER:
      (a) a final text answer, OR
      (b) a tool call:  get_weather(city="Chennai")

3. If it's a tool call:
      - YOUR code runs get_weather("Chennai")
      - you get a result:  {"temp": 31, "condition": "sunny"}

4. You send the result BACK to the model
   (as a "tool" role message).

5. The model now has the real data and either:
      - answers, OR
      - calls another tool (loop again).

6. Repeat until the model returns a final answer.
```

As a diagram:

```
        ┌─────────────────────────────┐
        ▼                             │
   User question                      │ (tool result
        ↓                             │  fed back)
       LLM ──── tool call? ──yes──▶ run tool ──┘
        │
        no
        ↓
    Final answer
```

This loop — call a tool, observe the result, decide again — is the seed of an **agent** (Phase 9). The only difference is how many times it loops and how it decides.

### Tool 1 — Calculator

LLMs are bad at exact arithmetic. A calculator tool fixes that.

```python
def calculator(expression: str) -> float:
    # In real code, use a safe evaluator, NOT raw eval()
    return safe_eval(expression)
```

Tool description given to the model:

```
name:        calculator
description: Evaluate a math expression and return the number.
parameters:  { expression: string }
```

Flow:

```
User:  "What is 1834 * 729?"
  ↓
LLM:   calculator(expression="1834 * 729")
  ↓
Code:  1337286
  ↓
LLM:   "1834 × 729 = 1,337,286."
```

Without the tool, the model might confidently return a **wrong** number. With it, the answer is exact.

### Tool 2 — Weather (external API)

This gives the model **real-time data** it could never know from training.

```python
def get_weather(city: str) -> dict:
    resp = requests.get(f"https://api.weather.com/v1/{city}")
    return resp.json()   # {"temp": 31, "condition": "sunny"}
```

Flow:

```
User:  "Should I carry an umbrella in London today?"
  ↓
LLM:   get_weather(city="London")
  ↓
Code:  {"temp": 14, "condition": "rain"}
  ↓
LLM:   "Yes — it's raining in London right now (14°C)."
```

This is the canonical tool-calling example because it shows the model fetching a fact, then **reasoning over the result** to give advice.

### Tool 3 — Database Lookup

This is where your backend skills shine: let the model query **your** data (safely).

```python
def get_customer_orders(customer_id: int) -> list:
    return db.query(
        "SELECT id, total FROM orders WHERE customer_id = %s",
        [customer_id],
    )
```

Flow:

```
User:  "How many orders has customer 42 placed?"
  ↓
LLM:   get_customer_orders(customer_id=42)
  ↓
Code:  [ {id: 1, total: 500}, {id: 2, total: 250} ]
  ↓
LLM:   "Customer 42 has placed 2 orders, totalling 750."
```

#### Safety rules (critical)

```
✓ Use parameterized queries (the model fills PARAMETERS, not raw SQL)
✓ Never let the model write arbitrary SQL it can execute
✓ Apply the same auth/permission checks as a normal endpoint
✓ Scope by user — recall prompt injection (Phase 5)
```

A tool is real code with real power. Treat a model-triggered tool call with the **same suspicion** as untrusted user input.

### Many Tools — How Does the Model Choose?

Give the model several tools and it selects based on each tool's **description** and the user's intent.

```
Available tools:
  - calculator         (math)
  - get_weather        (weather by city)
  - get_customer_orders(orders by customer id)

User: "What's the total of customer 42's orders, times 1.18 for tax?"

Model plan:
  1. get_customer_orders(42)   → 750
  2. calculator("750 * 1.18")  → 885
  3. answer: "With 18% tax, that's 885."
```

This multi-step selection is exactly **tool selection** + **agent loop**, which we formalize in Phase 9.

```
More tools  →  more power, but also:
  - higher chance of picking the wrong tool
  - clear descriptions matter even more
  - keep the toolset small and purposeful
```

### Summary & Goal

```
Tool        =  a function + a description the model can read
Tool call   =  the model asks to run a tool with arguments
The loop    =  ask → run tool → feed result back → repeat → answer
Your code   =  the thing that actually executes tools (safely)
```

#### Goal

***Understand that an agent is basically an LLM that can use tools.***

You now have an LLM that can:

```
- reason about a question
- choose a tool
- observe the result
- decide what to do next
```